# Circadian Metrics: IS, IV, and RA

This notebook is a self-contained version of the analysis. It can download the repo data, load the surgery metadata, compute per-session circadian metrics, and generate the same summary tables and plots without calling `Circadian_Metrics_IV_IS_RA.py`.

In [ ]:
import re
import shutil
import subprocess
import tempfile
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import expit, logit

try:
    import statsmodels.formula.api as smf
    HAS_STATSMODELS = True
except Exception:
    smf = None
    HAS_STATSMODELS = False

## Repo And Data Setup

Set the GitHub repo URL and the data folder to use from the repo. If the repo is already available in the notebook environment, the setup cell will reuse it.

In [ ]:
REPO_URL = "https://github.com/Oscar-Pineda524/CAAP193.git"
BRANCH_CANDIDATES = ["main", "master"]
NOTEBOOK_ROOT = Path.cwd()
REPO_DIR = NOTEBOOK_ROOT / "CAAP193_repo"
DATA_SUBDIR = "60s"
FORCE_REFRESH = False

Y_LIMITS = {
    "IS": (0.0, 0.7),
    "IV": (0.0, 0.6),
    "RA": (0.0, 1.0),
}

In [ ]:
def ensure_repo(repo_url: str, repo_dir: Path, branches: list[str], force_refresh: bool = False) -> Path:
    repo_dir = repo_dir.resolve()

    if force_refresh and repo_dir.exists():
        shutil.rmtree(repo_dir)

    if repo_dir.exists():
        print(f"Using existing repo at {repo_dir}")
        return repo_dir

    repo_dir.parent.mkdir(parents=True, exist_ok=True)

    try:
        subprocess.run(
            ["git", "clone", repo_url, str(repo_dir)],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        print(f"Cloned repo to {repo_dir}")
        return repo_dir
    except Exception as clone_error:
        print(f"git clone failed: {clone_error}")
        print("Falling back to downloading a zip archive from GitHub...")

    for branch in branches:
        zip_url = repo_url.removesuffix(".git") + f"/archive/refs/heads/{branch}.zip"
        try:
            with tempfile.TemporaryDirectory() as tmpdir:
                zip_path = Path(tmpdir) / f"{branch}.zip"
                urllib.request.urlretrieve(zip_url, zip_path)

                with zipfile.ZipFile(zip_path) as zf:
                    zf.extractall(tmpdir)

                extracted_root = next(Path(tmpdir).glob("*"))
                shutil.move(str(extracted_root), str(repo_dir))
                print(f"Downloaded repo archive ({branch}) to {repo_dir}")
                return repo_dir
        except Exception as zip_error:
            print(f"Zip download failed for branch '{branch}': {zip_error}")

    raise RuntimeError("Could not fetch the repo by git clone or zip download.")


REPO_DIR = ensure_repo(REPO_URL, REPO_DIR, BRANCH_CANDIDATES, force_refresh=FORCE_REFRESH)
DATA_ROOT = REPO_DIR / DATA_SUBDIR
SURGERY_FILE = REPO_DIR / "surgery_dates.csv"
OUTPUT_ROOT = REPO_DIR / f"scatter_plots_{DATA_ROOT.name}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repo directory: {REPO_DIR}")
print(f"Data directory: {DATA_ROOT}")
print(f"Surgery file: {SURGERY_FILE}")
print(f"Output directory: {OUTPUT_ROOT}")

## Metadata And Helper Functions

This section loads the surgery dates, maps each subject or subject initial to metadata, and defines the helper functions used throughout the analysis.

In [ ]:
surgery_map = {}
surgery_initial_map = {}

if not SURGERY_FILE.exists():
    raise FileNotFoundError(f"Could not find surgery metadata: {SURGERY_FILE}")

df_dates = pd.read_csv(SURGERY_FILE)
df_dates.columns = df_dates.columns.str.strip().str.lower()

required_cols = {"subject", "surgery_date"}
missing_cols = required_cols - set(df_dates.columns)
if missing_cols:
    raise ValueError(
        f"surgery_dates.csv is missing required columns: {sorted(missing_cols)}. "
        f"Found columns: {list(df_dates.columns)}"
    )

for _, row in df_dates.iterrows():
    subject = str(row["subject"]).strip()
    if not subject:
        continue

    surgery_date = pd.to_datetime(row["surgery_date"], format="%m/%d/%Y", errors="coerce")
    if pd.isna(surgery_date):
        raise ValueError(f"Invalid surgery date for subject '{subject}'")

    group = str(row.get("group", "Unknown")).strip().upper()
    if group not in {"T", "C"}:
        group = "Unknown"

    record = {
        "subject": subject,
        "subject_initial": subject[0].upper(),
        "surgery_date": surgery_date,
        "group": group,
    }
    surgery_map[subject] = record
    surgery_initial_map[record["subject_initial"]] = record


def parse_subject_and_date_from_filename(path: Path):
    fname = path.name
    m_date = re.search(r"(\d{4}-\d{2}-\d{2})", fname)
    session_date = pd.to_datetime(m_date.group(1), errors="coerce") if m_date else pd.NaT

    m_old = re.search(r"^(.*?)\s+(\d+)\s+\(\d{4}-\d{2}-\d{2}\)", fname)
    if m_old:
        label = m_old.group(1).strip()
        subj_id = m_old.group(2).strip()
        return label, subj_id, session_date, fname

    label = None
    subj_id = None
    if m_date:
        prefix = fname[:m_date.start()]
        prefix = re.sub(r"[\s_\-\(\)]+$", "", prefix)
        prefix = re.sub(r"^[\s_\-\(\)]+", "", prefix)
        label = prefix.strip() if prefix.strip() else None

        m_id = re.search(r"(\d+)", prefix)
        subj_id = m_id.group(1) if m_id else None

        if label:
            label_clean = re.sub(r"[_\-]*\d+[_\-]*", "", label).strip()
            label = label_clean if label_clean else label

    return label, subj_id, session_date, fname


def parse_subject_and_group_from_folder(folder_name: str):
    m = re.search(r"^(.*?)\((T|C)\)", folder_name)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    return folder_name.strip(), "Unknown"


def resolve_subject_metadata(subject_hint: str, group_hint: str = "Unknown"):
    subject_hint = str(subject_hint).strip()
    group_hint = str(group_hint).strip().upper()
    if group_hint not in {"T", "C"}:
        group_hint = "Unknown"

    candidates = []
    if subject_hint:
        candidates.append(subject_hint)
        candidates.append(subject_hint.upper())
        candidates.append(subject_hint.capitalize())
        candidates.append(subject_hint[:1].upper())

    for candidate in candidates:
        if candidate in surgery_map:
            record = surgery_map[candidate].copy()
            if record["group"] == "Unknown" and group_hint in {"T", "C"}:
                record["group"] = group_hint
            return record
        if len(candidate) == 1 and candidate in surgery_initial_map:
            record = surgery_initial_map[candidate].copy()
            if record["group"] == "Unknown" and group_hint in {"T", "C"}:
                record["group"] = group_hint
            return record

    raise ValueError(
        f"No surgery date found for subject '{subject_hint}' in surgery_dates.csv. "
        "The subject can be listed as the full name or its initial."
    )


def parse_subject_key_from_filename(path: Path):
    label, _, _, _ = parse_subject_and_date_from_filename(path)
    if label:
        cleaned = re.sub(r"[^A-Za-z]", "", str(label)).strip()
        if cleaned:
            return cleaned.upper()

    stem = path.stem
    m = re.search(r"([A-Za-z])", stem)
    if m:
        return m.group(1).upper()
    return None


def load_actilife_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    if "Datetime" in df.columns:
        df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")
    elif "datetime" in df.columns:
        df["Datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    elif "Date" in df.columns and "Time" in df.columns:
        df["Datetime"] = pd.to_datetime(
            df["Date"].astype(str) + " " + df["Time"].astype(str),
            errors="coerce",
        )
    else:
        raise ValueError(
            f"{path.name}: expected 'Datetime' column or ('Date' and 'Time'). "
            f"Found columns: {list(df.columns)}"
        )

    df = df.dropna(subset=["Datetime"]).sort_values("Datetime")

    if all(c in df.columns for c in ["accelerometer_x_sum", "accelerometer_y_sum", "accelerometer_z_sum"]):
        axis_cols = ["accelerometer_x_sum", "accelerometer_y_sum", "accelerometer_z_sum"]
    elif all(c in df.columns for c in ["Axis1", "Axis2", "Axis3"]):
        axis_cols = ["Axis1", "Axis2", "Axis3"]
    else:
        raise ValueError(
            f"{path.name}: expected accelerometer_x/y/z_sum or Axis1/2/3 columns. "
            f"Found columns: {list(df.columns)}"
        )

    df["MeanAxis"] = df[axis_cols].mean(axis=1)
    return df


def safe_div(num, den):
    if den is None or (isinstance(den, float) and np.isnan(den)) or den == 0:
        return np.nan
    return num / den


def compute_is_iv_ra(df: pd.DataFrame):
    hourly = df.set_index("Datetime")["MeanAxis"].resample("1h").mean().dropna()
    if len(hourly) < 24 or np.var(hourly.values, ddof=0) == 0:
        is_value = np.nan
    else:
        hourly_profile = hourly.groupby(hourly.index.hour).mean()
        is_value = safe_div(np.var(hourly_profile.values, ddof=0), np.var(hourly.values, ddof=0))

    x = df["MeanAxis"].to_numpy()
    vx = np.var(x, ddof=0)
    if len(x) < 2 or vx == 0:
        iv_value = np.nan
    else:
        iv_value = safe_div(np.mean(np.diff(x) ** 2), vx)

    if len(hourly) < 10:
        ra_value = np.nan
    else:
        m10_series = hourly.rolling(window=10, min_periods=10).mean()
        l5_series = hourly.rolling(window=5, min_periods=5).mean()
        m10 = m10_series.max(skipna=True)
        l5 = l5_series.min(skipna=True)

        if pd.isna(m10) or pd.isna(l5):
            ra_value = np.nan
        else:
            ra_value = safe_div((m10 - l5), (m10 + l5))

    return is_value, iv_value, ra_value


def compute_slope(x_vals: pd.Series, y_vals: pd.Series) -> float:
    mask = (~pd.isna(x_vals)) & (~pd.isna(y_vals))
    if mask.sum() < 2:
        return np.nan
    x = x_vals[mask].to_numpy(dtype=float)
    y = y_vals[mask].to_numpy(dtype=float)
    slope, _intercept = np.polyfit(x, y, 1)
    return float(slope)


def normalize_at_day(df: pd.DataFrame, metric: str, day: float = 60):
    x_vals = df["days_from_surgery"]
    y_vals = df[metric]
    mask = (~pd.isna(x_vals)) & (~pd.isna(y_vals))
    if mask.sum() < 2:
        return pd.Series([np.nan] * len(df), index=df.index)

    x = x_vals[mask].to_numpy(dtype=float)
    y = y_vals[mask].to_numpy(dtype=float)
    slope, intercept = np.polyfit(x, y, 1)
    value_at_day = slope * day + intercept
    return df[metric] - value_at_day


def add_trend_line(ax, x_vals, y_vals):
    mask = (~pd.isna(x_vals)) & (~pd.isna(y_vals))
    if mask.sum() < 2:
        return
    x = pd.Series(x_vals)[mask].to_numpy(dtype=float)
    y = pd.Series(y_vals)[mask].to_numpy(dtype=float)
    coeffs = np.polyfit(x, y, 1)
    y_fit = np.polyval(coeffs, x)
    order = np.argsort(x)
    ax.plot(x[order], y_fit[order], linestyle="--", linewidth=2)


df_dates

## Mixed Model Plot Helpers

These helpers generate the across-subject model plots after the per-subject processing is complete.

In [ ]:
def fit_mixedlm_and_plot(master_sessions_df: pd.DataFrame, metric: str, ylabel: str, title: str, out_name: str, plot_root: Path):
    df = master_sessions_df.copy()
    df = df.dropna(subset=["subject", "group", "days_from_surgery", metric])
    df = df[df["group"].isin(["T", "C"])].copy()
    if df.empty:
        print(f"[MIXEDLM] No data available for {metric}; skipping mixed model plot.")
        return

    df["days_from_surgery"] = df["days_from_surgery"].astype(float)
    df["group"] = pd.Categorical(df["group"], categories=["C", "T"])

    subjects = sorted(df["subject"].unique().tolist())
    cmap = plt.get_cmap("tab20")
    subj_to_color = {s: cmap(i % 20) for i, s in enumerate(subjects)}

    fig, ax = plt.subplots(figsize=(9, 5))
    for subject in subjects:
        subject_df = df[df["subject"] == subject]
        ax.scatter(subject_df["days_from_surgery"], subject_df[metric], label=subject, color=subj_to_color[subject], alpha=0.9)

        if len(subject_df) >= 2:
            slope = np.polyfit(subject_df["days_from_surgery"].to_numpy(), subject_df[metric].to_numpy(), 1)[0]
            xs = np.array([subject_df["days_from_surgery"].min(), subject_df["days_from_surgery"].max()])
            ys = np.polyval([slope, np.mean(subject_df[metric]) - slope * np.mean(subject_df["days_from_surgery"])], xs)
            ax.plot(xs, ys, color=subj_to_color[subject], linewidth=1, alpha=0.5)

    if HAS_STATSMODELS:
        try:
            model = smf.mixedlm(f"{metric} ~ days_from_surgery * group", df, groups=df["subject"], re_formula="~days_from_surgery")
            res = model.fit(reml=False, method="lbfgs", disp=False)
            print(f"[MIXEDLM] {metric} fit complete.")
            print(res.summary())

            x_grid = np.linspace(df["days_from_surgery"].min(), df["days_from_surgery"].max(), 200)
            for group in ["C", "T"]:
                pred_df = pd.DataFrame({"days_from_surgery": x_grid, "group": group, "subject": subjects[0]})
                y_hat = res.predict(pred_df)
                ax.plot(x_grid, y_hat, linewidth=3, alpha=0.9, label=f"MixedLM fit ({group})")
        except Exception as exc:
            print(f"[MIXEDLM] Could not fit mixed model for {metric}: {exc}")
    else:
        print(f"[MIXEDLM] statsmodels not available; skipping model fit for {metric}.")

    ax.set_xlabel("Days from surgery")
    ax.set_ylabel(ylabel)
    ax.set_title(title + "\n(All subjects; points colored by subject)")
    ax.axvline(0, linestyle=":", linewidth=1)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0.0, fontsize=8)
    plt.tight_layout()

    out_path = plot_root / out_name
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Mixed model plot saved: {out_path}")


def transform_metric_for_glmm(df: pd.DataFrame, metric: str):
    out = df.copy()
    eps = 1e-6
    resp_col = f"{metric}_glmm_resp"
    if metric in ["IS", "RA"]:
        out[resp_col] = logit(out[metric].clip(eps, 1 - eps))
    elif metric == "IV":
        out[resp_col] = np.log(out[metric].clip(eps))
    else:
        raise ValueError(f"Unsupported GLMM metric: {metric}")
    return out, resp_col


def inverse_transform_glmm_response(y_vals: np.ndarray, metric: str) -> np.ndarray:
    if metric in ["IS", "RA"]:
        return expit(y_vals)
    if metric == "IV":
        return np.exp(y_vals)
    raise ValueError(f"Unsupported GLMM metric: {metric}")


def fit_glmmish_and_plot(master_sessions_df: pd.DataFrame, metric: str, ylabel: str, title: str, out_name: str, plot_root: Path):
    df = master_sessions_df.copy()
    df = df.dropna(subset=["subject", "group", "days_from_surgery", metric])
    df = df[df["group"].isin(["T", "C"])].copy()
    if df.empty:
        print(f"[GLMM] No data available for {metric}; skipping approximate GLMM plot.")
        return

    df["days_from_surgery"] = df["days_from_surgery"].astype(float)
    df["group"] = pd.Categorical(df["group"], categories=["C", "T"])

    subjects = sorted(df["subject"].unique().tolist())
    cmap = plt.get_cmap("tab20")
    subj_to_color = {s: cmap(i % 20) for i, s in enumerate(subjects)}

    fig, ax = plt.subplots(figsize=(9, 5))
    for subject in subjects:
        subject_df = df[df["subject"] == subject]
        ax.scatter(subject_df["days_from_surgery"], subject_df[metric], label=subject, color=subj_to_color[subject], alpha=0.9)

    if HAS_STATSMODELS:
        try:
            model_df, resp_col = transform_metric_for_glmm(df, metric)
            model = smf.mixedlm(f"{resp_col} ~ days_from_surgery * group", model_df, groups=model_df["subject"], re_formula="~days_from_surgery")
            res = model.fit(reml=False, method="lbfgs", disp=False)
            print(f"[GLMM-approx] {metric} fit complete.")
            print(res.summary())

            summary_path = plot_root / f"glmm_{metric}_summary.txt"
            with open(summary_path, "w", encoding="utf-8") as handle:
                handle.write(str(res.summary()))
            print(f"Approximate GLMM summary saved: {summary_path}")

            x_grid = np.linspace(df["days_from_surgery"].min(), df["days_from_surgery"].max(), 200)
            for group in ["C", "T"]:
                pred_df = pd.DataFrame({"days_from_surgery": x_grid, "group": group, "subject": subjects[0]})
                y_hat_trans = res.predict(pred_df)
                y_hat_orig = inverse_transform_glmm_response(np.asarray(y_hat_trans), metric)
                ax.plot(x_grid, y_hat_orig, linewidth=3, alpha=0.95, label=f"Approx. GLMM fit ({group})")
        except Exception as exc:
            print(f"[GLMM] Could not fit approximate GLMM for {metric}: {exc}")
    else:
        print(f"[GLMM] statsmodels not available; skipping approximate GLMM fit for {metric}.")

    ax.set_xlabel("Days from surgery")
    ax.set_ylabel(ylabel)
    ax.set_title(title + "\n(All subjects; points colored by subject)")
    ax.axvline(0, linestyle=":", linewidth=1)
    if metric in Y_LIMITS:
        ax.set_ylim(Y_LIMITS[metric])
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0.0, fontsize=8)
    plt.tight_layout()

    out_path = plot_root / out_name
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Approximate GLMM plot saved: {out_path}")

## Run The Analysis

This cell discovers the available input files, computes the per-session and per-subject summaries, and saves all outputs.

In [ ]:
if not DATA_ROOT.exists() or not DATA_ROOT.is_dir():
    raise ValueError(f"Provided data directory is invalid: {DATA_ROOT}")

subject_dirs = sorted([d for d in DATA_ROOT.iterdir() if d.is_dir()])
root_csv_files = sorted(DATA_ROOT.glob("*.csv"))
subject_batches = []

if subject_dirs:
    print(f"Found {len(subject_dirs)} subject folders in: {DATA_ROOT}\n")
    for subj_dir in subject_dirs:
        folder_subject, folder_group = parse_subject_and_group_from_folder(subj_dir.name)
        try:
            metadata = resolve_subject_metadata(folder_subject, folder_group)
        except ValueError as exc:
            print(f"  [WARN] {exc} Skipping folder '{subj_dir.name}'.")
            continue

        subject_batches.append({
            "source_name": subj_dir.name,
            "subject": metadata["subject"],
            "group": metadata["group"],
            "surgery_date": metadata["surgery_date"],
            "paths": sorted(subj_dir.glob("*.csv")),
            "plot_dir_name": subj_dir.name,
        })
elif root_csv_files:
    print(f"Found {len(root_csv_files)} CSV files directly inside: {DATA_ROOT}\n")
    grouped_paths = {}
    for path in root_csv_files:
        subject_key = parse_subject_key_from_filename(path)
        if not subject_key:
            print(f"  [WARN] Could not determine subject initial from filename: {path.name}")
            continue
        grouped_paths.setdefault(subject_key, []).append(path)

    for subject_key in sorted(grouped_paths):
        try:
            metadata = resolve_subject_metadata(subject_key)
        except ValueError as exc:
            print(f"  [WARN] {exc} Skipping files for initial '{subject_key}'.")
            continue

        subject_batches.append({
            "source_name": subject_key,
            "subject": metadata["subject"],
            "group": metadata["group"],
            "surgery_date": metadata["surgery_date"],
            "paths": sorted(grouped_paths[subject_key]),
            "plot_dir_name": metadata["subject"],
        })
else:
    raise ValueError(f"No CSV files or subject subfolders found inside: {DATA_ROOT}")

master_summary_rows = []
master_sessions_rows = []

for batch in subject_batches:
    print("=" * 60)
    print(f"Subject source: {batch['source_name']}")

    subject_name = batch["subject"]
    group_label = batch["group"]
    surgery_date = batch["surgery_date"]
    paths = batch["paths"]

    if not paths:
        print("  [SKIP] No CSV files found for this subject.")
        continue

    print(f"  Found {len(paths)} CSV files")
    for path in paths[:10]:
        print("   -", path.name)
    if len(paths) > 10:
        print("   ...")

    plot_dir = OUTPUT_ROOT / batch["plot_dir_name"]
    plot_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for path in paths:
        label, subj_id, session_date, fname = parse_subject_and_date_from_filename(path)
        try:
            df = load_actilife_csv(path)
        except Exception as exc:
            print(f"  [WARN] Could not load {path.name}: {exc}")
            continue

        is_value, iv_value, ra_value = compute_is_iv_ra(df)
        rows.append({
            "subject": subject_name,
            "id": subj_id,
            "group": group_label,
            "session_date": session_date.date() if pd.notna(session_date) else None,
            "file": fname,
            "IS": is_value,
            "IV": iv_value,
            "RA": ra_value,
            "n_minutes": len(df),
        })

    if not rows:
        print("  [SKIP] No usable CSV files after loading/processing.")
        continue

    per_session = pd.DataFrame(rows)
    per_session["session_date"] = pd.to_datetime(per_session["session_date"], errors="coerce")
    per_session = per_session.dropna(subset=["session_date"]).copy()
    per_session["days_from_surgery"] = (per_session["session_date"] - surgery_date).dt.days
    per_session = per_session.sort_values("session_date").reset_index(drop=True)

    per_session["IS_norm60"] = normalize_at_day(per_session, "IS", 60)
    per_session["IV_norm60"] = normalize_at_day(per_session, "IV", 60)
    per_session["RA_norm60"] = normalize_at_day(per_session, "RA", 60)

    if per_session.empty:
        print("  [SKIP] All sessions were dropped because no YYYY-MM-DD date was found in filenames.")
        continue

    is_slope = compute_slope(per_session["days_from_surgery"], per_session["IS"])
    iv_slope = compute_slope(per_session["days_from_surgery"], per_session["IV"])
    ra_slope = compute_slope(per_session["days_from_surgery"], per_session["RA"])

    summary_row = {
        "folder": batch["source_name"],
        "subject": subject_name,
        "group": group_label,
        "surgery_date": str(surgery_date.date()),
        "n_sessions": int(per_session["days_from_surgery"].notna().sum()),
        "IS_mean": per_session["IS"].mean(skipna=True),
        "IS_sd": per_session["IS"].std(skipna=True),
        "IV_mean": per_session["IV"].mean(skipna=True),
        "IV_sd": per_session["IV"].std(skipna=True),
        "RA_mean": per_session["RA"].mean(skipna=True),
        "RA_sd": per_session["RA"].std(skipna=True),
        "IS_slope": is_slope,
        "IV_slope": iv_slope,
        "RA_slope": ra_slope,
    }
    master_summary_rows.append(summary_row)

    print("\nPer-session circadian metrics (this subject):")
    display(per_session)
    print("\nGrouped longitudinal summary (this subject):")
    display(pd.DataFrame([summary_row]))

    per_session_out = plot_dir / "per_session_metrics.csv"
    per_session.to_csv(per_session_out, index=False)

    for _, row in per_session.iterrows():
        master_sessions_rows.append({
            "subject": subject_name,
            "group": group_label,
            "surgery_date": str(surgery_date.date()),
            "session_date": row["session_date"],
            "days_from_surgery": row["days_from_surgery"],
            "IS": row["IS"],
            "IV": row["IV"],
            "RA": row["RA"],
            "IS_norm60": row["IS_norm60"],
            "IV_norm60": row["IV_norm60"],
            "RA_norm60": row["RA_norm60"],
        })

    for metric, ylabel, title in [
        ("IS", "Interdaily Stability (IS)", "Longitudinal IS across sessions"),
        ("IV", "Intradaily Variability (IV)", "Longitudinal IV across sessions"),
        ("RA", "Relative Amplitude (RA, classic M10/L5)", "Longitudinal RA across sessions"),
    ]:
        fig, ax = plt.subplots(figsize=(9, 4))
        color = "red" if group_label == "T" else "blue" if group_label == "C" else "gray"
        ax.scatter(per_session["days_from_surgery"], per_session[metric], color=color)
        add_trend_line(ax, per_session["days_from_surgery"], per_session[metric])
        for line in ax.get_lines():
            line.set_color(color)
        ax.set_xlabel("Days from surgery")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{title}\nSubject: {subject_name}  |  Group: {group_label}  |  Surgery: {surgery_date.date()}")
        if metric in Y_LIMITS:
            ax.set_ylim(Y_LIMITS[metric])
        ax.axvline(0, linestyle=":", linewidth=1)
        plt.tight_layout()
        out_path = plot_dir / f"{metric}_scatter_trend.png"
        plt.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close()
        print(f"  Saved {out_path}")

    for metric, ylabel, title in [
        ("IS_norm60", "Normalized IS (relative to day 60)", "Normalized IS across sessions"),
        ("IV_norm60", "Normalized IV (relative to day 60)", "Normalized IV across sessions"),
        ("RA_norm60", "Normalized RA (classic M10/L5, relative to day 60)", "Normalized RA across sessions"),
    ]:
        fig, ax = plt.subplots(figsize=(9, 4))
        color = "red" if group_label == "T" else "blue" if group_label == "C" else "gray"
        ax.scatter(per_session["days_from_surgery"], per_session[metric], color=color)
        add_trend_line(ax, per_session["days_from_surgery"], per_session[metric])
        for line in ax.get_lines():
            line.set_color(color)
        ax.set_xlabel("Days from surgery")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{title}\nSubject: {subject_name}  |  Group: {group_label}  |  Surgery: {surgery_date.date()}")
        ax.axvline(0, linestyle=":", linewidth=1)
        ax.axhline(0, linestyle="--", linewidth=1)
        plt.tight_layout()
        out_path = plot_dir / f"{metric}_scatter_trend.png"
        plt.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close()
        print(f"  Saved {out_path}")

print("\n" + "=" * 60)
print("DONE. Outputs saved under:")
print(f"  {OUTPUT_ROOT}")

if master_summary_rows:
    master_df = pd.DataFrame(master_summary_rows).sort_values(["group", "subject"])
    master_out = OUTPUT_ROOT / "master_summary.csv"
    master_df.to_csv(master_out, index=False)
    print(f"Master summary saved: {master_out}")
else:
    print("No subjects were processed successfully; no master summary written.")

if master_summary_rows:
    master_df = pd.DataFrame(master_summary_rows)
    master_df = master_df[master_df["group"].isin(["T", "C"])].copy()

    def plot_group_slope(metric_key: str, title: str, ylabel: str, out_name: str):
        sub = master_df.dropna(subset=[metric_key])
        if sub.empty:
            print(f"No data available for {metric_key}; skipping plot.")
            return
        grp = sub.groupby("group")[metric_key]
        means = grp.mean()
        sems = grp.sem()
        order = [group for group in ["T", "C"] if group in means.index.tolist()]
        means = means.loc[order]
        sems = sems.loc[order]
        fig, ax = plt.subplots(figsize=(5.5, 4))
        ax.bar(order, means.values, yerr=sems.values, capsize=6)
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.set_xlabel("Group")
        ax.axhline(0, linewidth=1)
        plt.tight_layout()
        out_path = OUTPUT_ROOT / out_name
        plt.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close()
        print(f"Group comparison plot saved: {out_path}")

    plot_group_slope("IS_slope", "Mean slope of IS vs Days from surgery (T vs C)", "IS slope (per day)", "compare_IS_slope_T_vs_C.png")
    plot_group_slope("IV_slope", "Mean slope of IV vs Days from surgery (T vs C)", "IV slope (per day)", "compare_IV_slope_T_vs_C.png")
    plot_group_slope("RA_slope", "Mean slope of RA vs Days from surgery (T vs C)", "RA slope (per day)", "compare_RA_slope_T_vs_C.png")

if master_sessions_rows:
    master_sessions_df = pd.DataFrame(master_sessions_rows)
    mixed_out = OUTPUT_ROOT / "mixed_model_plots"
    mixed_out.mkdir(parents=True, exist_ok=True)

    fit_mixedlm_and_plot(master_sessions_df, "IS", "Interdaily Stability (IS)", "IS vs Days from surgery", "mixed_IS_all_subjects.png", mixed_out)
    fit_mixedlm_and_plot(master_sessions_df, "IV", "Intradaily Variability (IV)", "IV vs Days from surgery", "mixed_IV_all_subjects.png", mixed_out)
    fit_mixedlm_and_plot(master_sessions_df, "RA", "Relative Amplitude (RA, classic M10/L5)", "RA vs Days from surgery", "mixed_RA_all_subjects.png", mixed_out)
    fit_mixedlm_and_plot(master_sessions_df, "IS_norm60", "Normalized IS (relative to day 60)", "Normalized IS vs Days from surgery", "mixed_IS_norm60_all_subjects.png", mixed_out)
    fit_mixedlm_and_plot(master_sessions_df, "IV_norm60", "Normalized IV (relative to day 60)", "mixed_IV_norm60_all_subjects.png", mixed_out)
    fit_mixedlm_and_plot(master_sessions_df, "RA_norm60", "Normalized RA (classic M10/L5, relative to day 60)", "mixed_RA_norm60_all_subjects.png", mixed_out)

    master_sessions_out = mixed_out / "master_sessions_long.csv"
    master_sessions_df.to_csv(master_sessions_out, index=False)
    print(f"Master per-session long table saved: {master_sessions_out}")
else:
    print("No per-session rows collected; skipping mixed model plots.")

if master_sessions_rows:
    master_sessions_df = pd.DataFrame(master_sessions_rows)
    glmm_out = OUTPUT_ROOT / "glmm_plots"
    glmm_out.mkdir(parents=True, exist_ok=True)

    fit_glmmish_and_plot(master_sessions_df, "IS", "Interdaily Stability (IS)", "Approximate GLMM for IS vs Days from surgery", "glmm_IS_all_subjects.png", glmm_out)
    fit_glmmish_and_plot(master_sessions_df, "IV", "Intradaily Variability (IV)", "Approximate GLMM for IV vs Days from surgery", "glmm_IV_all_subjects.png", glmm_out)
    fit_glmmish_and_plot(master_sessions_df, "RA", "Relative Amplitude (RA, classic M10/L5)", "Approximate GLMM for RA vs Days from surgery", "glmm_RA_all_subjects.png", glmm_out)
else:
    print("No per-session rows collected; skipping approximate GLMM plots.")

## Notes

- Add markdown cells anywhere in the notebook to explain the analysis.
- Change `DATA_SUBDIR` if the actigraphy files live in a different repo folder.
- Set `FORCE_REFRESH = True` if you want the notebook to download a fresh copy of the repo.